# Лабораторная работа №2
## «Виртуальный датчик для контроля процесса обжига в печи»

**Задача:** построить модель (soft sensor), которая прогнозирует концентрацию выходного продукта в реальном времени на основе минутной телеметрии, без ожидания лабораторного анализа.

**Данные:**
- `data_train.csv` — минутные измерения 16 показателей телеметрии за ~7 месяцев
- `target_train.csv` — нерегулярные лабораторные замеры с задержкой 10–15 мин
- `data_test_small.csv`, `target_test_small.csv` — аналогичные тестовые данные

In [ ]:
# Установка зависимостей (раскомментировать при необходимости)
# !pip install pandas==2.1.4 numpy==1.26.4 matplotlib==3.8.3 seaborn==0.13.2 \
#             scipy==1.12.0 statsmodels==0.14.1 lightgbm==4.3.0 \
#             xgboost==2.0.3 scikit-learn==1.4.2 shap==0.45.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('Библиотеки успешно импортированы')

---
## 2.1 Разведочный анализ данных (EDA)

In [ ]:
# ------------------------------------------------------------
# 1.1 Загрузка данных
# ------------------------------------------------------------

data_train = pd.read_csv('data_train.csv', parse_dates=['datetime'], index_col='datetime')

target_train = pd.read_csv('target_train.csv', parse_dates=['Дата'], index_col='Дата')
target_train.index.name = 'datetime'
target_train.columns = ['target']

data_test = pd.read_csv('data_test_small.csv', parse_dates=['datetime'], index_col='datetime')
target_test = pd.read_csv('target_test_small.csv', parse_dates=['Дата'], index_col='Дата')
target_test.index.name = 'datetime'
target_test.columns = ['target']

print('=== Телеметрия (train) ===')
print(f'Период: {data_train.index.min()} → {data_train.index.max()}')
print(f'Форма: {data_train.shape}')
print(f'Пропуски:\n{data_train.isnull().sum()}')
print('\n=== Целевая переменная (train) ===')
print(f'Период: {target_train.index.min()} → {target_train.index.max()}')
print(f'Форма: {target_train.shape}')
print(f'Пропуски: {target_train.isnull().sum().values[0]}')

**Наблюдение по пропускам:** telemetry_0–11 имеют единичные пропуски (<0.05%) — штатная телеметрия. Telemetry_12–15 заполнены лишь на ~0.6% (появились только в конце периода) — исключаем из обучения.

In [ ]:
# ------------------------------------------------------------
# 1.2 Визуализация всех признаков телеметрии
# ------------------------------------------------------------

fig, axes = plt.subplots(8, 2, figsize=(16, 32))
axes = axes.flatten()

for i, col in enumerate(data_train.columns):
    ax = axes[i]
    sample = data_train[col].dropna().iloc[:10_000]
    ax.plot(sample.index, sample.values, linewidth=0.5, alpha=0.8)
    ax.set_title(col)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Телеметрия — первые 10 000 минут (train)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 1.3 Анализ целевой переменной
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(target_train.index, target_train['target'], 'o-', markersize=3, linewidth=1)
axes[0].set_title('Целевая переменная во времени')
axes[0].set_xlabel('Дата')
axes[0].set_ylabel('target')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
axes[0].tick_params(axis='x', rotation=30)

axes[1].hist(target_train['target'], bins=40, edgecolor='black', alpha=0.7)
axes[1].set_title('Распределение target')
axes[1].set_xlabel('target')
axes[1].set_ylabel('Частота')

axes[2].boxplot(target_train['target'].dropna())
axes[2].set_title('Box-plot target')
axes[2].set_ylabel('target')

plt.tight_layout()
plt.show()

print('=== Статистика целевой переменной ===')
print(target_train['target'].describe().round(4))

gaps = target_train.index.to_series().diff().dropna()
print(f'\nМедианный интервал между замерами: {gaps.median()}')
print(f'Мин. интервал: {gaps.min()}')
print(f'Макс. интервал: {gaps.max()}')

In [ ]:
# ------------------------------------------------------------
# 1.4 Корреляции телеметрии с таргетом при разных лагах
# Стратегия: forward-fill таргета на минутную сетку для EDA
# ------------------------------------------------------------

TELE_COLS = [f'telemetry_{i}' for i in range(12)]  # только telemetry_0–11
target_minute = target_train.reindex(data_train.index).ffill()

corr_lags = {}
for lag in [0, 10, 30, 60, 120]:
    shifted = target_minute.shift(-lag)
    corr = data_train[TELE_COLS].corrwith(shifted['target']).rename(lag)
    corr_lags[lag] = corr

corr_df = pd.DataFrame(corr_lags)
corr_df.columns = [f'lag={c}m' for c in corr_df.columns]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, linewidths=0.5)
ax.set_title('Корреляция телеметрии с target при разных лагах')
plt.tight_layout()
plt.show()

### Выводы по EDA

1. **Пропуски:** telemetry_12–15 отсутствуют на 99.4% периода — исключены из обучения. Telemetry_4 имеет блоки пропусков >10 мин — будет заполнена медианой.
2. **Характер признаков:** три группы — стабильные (telemetry_4, telemetry_8), медленно дрейфующие (telemetry_9–11), высокошумные (telemetry_1, 3, 5).
3. **Целевая переменная:** правосторонняя асимметрия, мода ~0.20–0.22, выбросы >0.5 соответствуют нештатным ситуациям. Медианный интервал замеров — 2 часа.
4. **Корреляции:** слабые (|r| ≤ 0.18), нелинейные модели обоснованы. Наиболее информативны telemetry_9 (−0.17), telemetry_15 (+0.18), telemetry_10, telemetry_11 (−0.15).
5. **Динамика лагов:** корреляции почти не меняются при лаге 0–120 мин — система инерционна, окна в часы будут полезны.

---
## Синхронизация данных

Лабораторный замер в момент T соответствует состоянию печи в T−10..15 мин. Сдвигаем индекс таргета на 12 мин назад и применяем `merge_asof`.

In [ ]:
# ------------------------------------------------------------
# 2.1 Очистка телеметрии
# ------------------------------------------------------------

# Линейная интерполяция для малых пропусков (до 10 мин подряд)
data_train_clean = data_train[TELE_COLS].interpolate(method='linear', limit=10)

# Обрезка выбросов по перцентилям 0.5%–99.5%
for col in TELE_COLS:
    q_lo = data_train_clean[col].quantile(0.005)
    q_hi = data_train_clean[col].quantile(0.995)
    data_train_clean[col] = data_train_clean[col].clip(q_lo, q_hi)

# Telemetry_4: блоки пропусков >10 мин заполняем медианой
median_t4 = data_train_clean['telemetry_4'].median()
data_train_clean['telemetry_4'] = data_train_clean['telemetry_4'].fillna(median_t4)

print(f'Пропуски после обработки: {data_train_clean.isnull().sum().sum()}')
print(f'telemetry_4 заполнена медианой: {median_t4:.4f}')

In [ ]:
# ------------------------------------------------------------
# 2.2 Синхронизация merge_asof с учётом задержки 12 мин
# ------------------------------------------------------------

LAG_MINUTES = 12

# Сдвигаем индекс таргета назад — ищем телеметрию в момент взятия пробы
target_shifted = target_train.copy()
target_shifted.index = target_train.index - pd.Timedelta(minutes=LAG_MINUTES)

tele_reset   = data_train_clean.reset_index()
target_reset = target_shifted.reset_index()

df_sync = pd.merge_asof(
    target_reset.sort_values('datetime'),
    tele_reset.sort_values('datetime'),
    on='datetime',
    direction='backward',
    tolerance=pd.Timedelta('5min')
).set_index('datetime').dropna()

print(f'Синхронизированный датасет: {df_sync.shape}')
print(f'Период: {df_sync.index.min()} → {df_sync.index.max()}')
print(f'Потеряно точек (нет телеметрии рядом): {len(target_train) - len(df_sync)}')

gaps_sync = df_sync.index.to_series().diff().dropna()
print(f'Медианный интервал между замерами: {gaps_sync.median()}')

### Выводы по синхронизации

Потеряно 2 точки из 1773 (<0.1%) — синхронизация практически идеальна. Медианный интервал между замерами — ровно 2 часа (88% случаев). Это регулярный график отбора проб; 4-часовые интервалы (10%) — пропущенные плановые замеры.

---
## 2.2 Инжиниринг признаков

In [ ]:
# ------------------------------------------------------------
# 3.1 Ключевые признаки для feature engineering
# (только из TELE_COLS = telemetry_0–11)
# ------------------------------------------------------------

KEY_COLS = ['telemetry_9', 'telemetry_10', 'telemetry_11',
            'telemetry_8', 'telemetry_0', 'telemetry_3',
            'telemetry_5', 'telemetry_6']

# Окна скользящих статистик
WINDOWS = {
    '30min': 30,
    '2h':   120,
    '6h':   360,
    '12h':  720,
}

In [ ]:
# ------------------------------------------------------------
# 3.2 Скользящие статистики и признаки динамики
# Вычисляем на полной минутной телеметрии, затем merge_asof
# ------------------------------------------------------------

rolling_frames = []

for col in KEY_COLS:
    series = data_train_clean[col]
    # Скользящие статистики
    for win_name, win_size in WINDOWS.items():
        roll = series.rolling(window=win_size, min_periods=win_size//2)
        rolling_frames.append(roll.mean().rename(f'{col}_rmean_{win_name}'))
        rolling_frames.append(roll.std().rename(f'{col}_rstd_{win_name}'))
        rolling_frames.append(roll.min().rename(f'{col}_rmin_{win_name}'))
        rolling_frames.append(roll.max().rename(f'{col}_rmax_{win_name}'))
    # Признаки динамики
    rolling_frames.append(series.diff(1).rename(f'{col}_diff1'))
    rolling_frames.append(series.diff(10).rename(f'{col}_diff10'))
    rolling_frames.append(series.pct_change(30).replace(
        [np.inf, -np.inf], np.nan).rename(f'{col}_pct30'))

df_rolling = pd.concat(rolling_frames, axis=1)
print(f'Rolling + dynamic признаков: {df_rolling.shape[1]}')

In [ ]:
# ------------------------------------------------------------
# 3.3 Присоединение rolling к синхронизированным точкам
# ------------------------------------------------------------

target_reset2 = target_shifted.reset_index()

df_full = pd.merge_asof(
    target_reset2.sort_values('datetime'),
    pd.concat([data_train_clean, df_rolling], axis=1).reset_index().sort_values('datetime'),
    on='datetime',
    direction='backward',
    tolerance=pd.Timedelta('5min')
).set_index('datetime').dropna(subset=['target'])

df_full = df_full.sort_index()

In [ ]:
# ------------------------------------------------------------
# 3.4 Лаговые признаки (по уровню синхронизированных точек ≈ 2ч шаг)
# и авторегрессионные лаги таргета
# ------------------------------------------------------------

for col in KEY_COLS:
    for lag in [1, 2, 3, 6]:
        df_full[f'{col}_lag{lag}'] = df_full[col].shift(lag)

for lag in [1, 2, 3]:
    df_full[f'target_lag{lag}'] = df_full['target'].shift(lag)

# Временные признаки (циклическое кодирование часа)
df_full['hour']        = df_full.index.hour
df_full['day_of_week'] = df_full.index.dayofweek
df_full['month']       = df_full.index.month
df_full['hour_sin']    = np.sin(2 * np.pi * df_full['hour'] / 24)
df_full['hour_cos']    = np.cos(2 * np.pi * df_full['hour'] / 24)

n_before = len(df_full)
df_full  = df_full.dropna()
print(f'Строк до dropna: {n_before}, после: {len(df_full)}')
print(f'Итоговых признаков: {df_full.shape[1] - 1} (без target)')

In [ ]:
# ------------------------------------------------------------
# 3.5 Корреляция признаков с таргетом (топ-20)
# ------------------------------------------------------------

corr_with_target = df_full.corr()['target'].drop('target').abs().sort_values(ascending=False)

print('=== Топ-20 признаков по |корреляции| с target ===')
print(corr_with_target.head(20).round(4))

fig, ax = plt.subplots(figsize=(10, 8))
corr_with_target.head(25).sort_values().plot(kind='barh', ax=ax)
ax.set_title('Топ-25 признаков по |корреляции| с target')
ax.set_xlabel('|Pearson r|')
plt.tight_layout()
plt.show()

### Выводы по инжинирингу признаков

1. **target_lag1** (r=0.55) — доминирующий предиктор. Концентрация — инерционный процесс; текущее значение сильно зависит от предыдущего замера (2 ч назад).
2. **Rolling-минимумы telemetry_9** на окнах 2ч/6ч/12ч стабильно в топ-10 — важен не мгновенный показатель, а устойчиво низкий уровень за последние часы.
3. **telemetry_0_rmax_30min** (r=0.15) — краткосрочный максимум этого датчика предсказывает концентрацию лучше мгновенного значения.
4. **204 признака** при 1760 наблюдениях — приемлемо для деревьев; линейные модели потребуют регуляризации.

---
## 2.3 Построение прогнозных моделей

### Обоснование выбора и отказа от моделей

**Модели, которые точно НЕ подходят:**
- **ARIMA/SARIMA** — работают с одномерным равноотстоящим рядом. У нас 204 экзогенных признака и нерегулярные интервалы. ARIMAX формально возможен, но не масштабируется.
- **VECM / тест Йохансена** — предназначены для коинтеграции нескольких рядов одинаковой частоты. Таргет нерегулярен.
- **Prophet** — ориентирован на тренд и сезонность одномерного ряда. Нет выраженной сезонности; сигнал идёт от телеметрии.
- **TCN (нейросеть)** — требует регулярного ряда и значительно больше данных (1760 точек — мало).

**Выбранные модели:**
| Модель | Обоснование |
|--------|-------------|
| Ridge Regression | Линейный бейзлайн с L2-регуляризацией против мультиколлинеарности |
| LightGBM | Градиентный бустинг — лучший выбор для табличных данных с лагами |
| XGBoost | Сравнение с LightGBM; другая реализация бустинга |
| Random Forest | Ансамбль без склонности к переобучению; устойчив к шумным признакам |

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
import xgboost as xgb
import shap

# ------------------------------------------------------------
# 4.1 Хронологический train/test split (не случайный!)
# ------------------------------------------------------------

feature_cols = [c for c in df_full.columns if c != 'target']
X = df_full[feature_cols]
y = df_full['target']

split_idx  = int(len(df_full) * 0.85)
split_date = df_full.index[split_idx]

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {X_train.shape}, {X_train.index.min().date()} → {X_train.index.max().date()}')
print(f'Test:  {X_test.shape},  {X_test.index.min().date()} → {X_test.index.max().date()}')

In [ ]:
# ------------------------------------------------------------
# 4.2 Функция оценки метрик
# ------------------------------------------------------------

def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

def mape_safe(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps)))

def directional_accuracy(y_true, y_pred):
    """Доля правильно предсказанных направлений изменения"""
    true_dir = np.diff(y_true.values)
    pred_dir = np.diff(y_pred)
    return np.mean(np.sign(true_dir) == np.sign(pred_dir))

def evaluate(name, y_true, y_pred):
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mape  = mape_safe(y_true, y_pred)
    wape_ = wape(y_true.values, y_pred)
    da    = directional_accuracy(y_true, y_pred)
    print(f"\n{'='*45}")
    print(f'  {name}')
    print(f"{'='*45}")
    print(f'  MAE:  {mae:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  MAPE: {mape:.4f}')
    print(f'  WAPE: {wape_:.4f}')
    print(f'  Directional Accuracy: {da:.4f}')
    return {'model': name, 'MAE': mae, 'RMSE': rmse,
            'MAPE': mape, 'WAPE': wape_, 'DA': da}

In [ ]:
# ------------------------------------------------------------
# 4.3 Модель 1: Ridge Regression
# ------------------------------------------------------------

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

ridge = Ridge(alpha=10.0)
ridge.fit(X_train_sc, y_train)

y_pred_ridge  = ridge.predict(X_test_sc)
results_ridge = evaluate('Ridge Regression', y_test, y_pred_ridge)

In [ ]:
# ------------------------------------------------------------
# 4.4 Модель 2: LightGBM
# ------------------------------------------------------------

lgb_params = {
    'objective':        'regression',
    'metric':           'mae',
    'n_estimators':     500,
    'learning_rate':    0.05,
    'num_leaves':       31,
    'min_child_samples': 20,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     42,
    'verbose':         -1,
}

lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]
)

y_pred_lgb  = lgb_model.predict(X_test)
results_lgb = evaluate('LightGBM', y_test, y_pred_lgb)

In [ ]:
# ------------------------------------------------------------
# 4.5 Модель 3: XGBoost
# ------------------------------------------------------------

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=50,
    eval_metric='mae',
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_xgb  = xgb_model.predict(X_test)
results_xgb = evaluate('XGBoost', y_test, y_pred_xgb)

In [ ]:
# ------------------------------------------------------------
# 4.6 Модель 4: Random Forest
# ------------------------------------------------------------

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    max_features=0.5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred_rf  = rf_model.predict(X_test)
results_rf = evaluate('Random Forest', y_test, y_pred_rf)

In [ ]:
# ------------------------------------------------------------
# 4.7 Сводная таблица метрик
# ------------------------------------------------------------

results_df = pd.DataFrame([results_ridge, results_lgb, results_xgb, results_rf])
results_df = results_df.set_index('model').round(4)
print('\n=== СВОДНАЯ ТАБЛИЦА МЕТРИК ===')
print(results_df.to_string())

In [ ]:
# ------------------------------------------------------------
# 4.8 Визуализация прогнозов vs реальных значений
# ------------------------------------------------------------

fig, axes = plt.subplots(4, 1, figsize=(14, 16))
models_preds = [
    ('Ridge Regression', y_pred_ridge),
    ('LightGBM',         y_pred_lgb),
    ('XGBoost',          y_pred_xgb),
    ('Random Forest',    y_pred_rf),
]

for ax, (name, y_pred) in zip(axes, models_preds):
    ax.plot(y_test.index, y_test.values, 'o-', markersize=3,
            label='Факт', color='steelblue', linewidth=1)
    ax.plot(y_test.index, y_pred, 's--', markersize=3,
            label='Прогноз', color='tomato', linewidth=1, alpha=0.8)
    ax.set_title(name)
    ax.legend(loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.set_ylabel('target')

plt.suptitle('Прогноз vs Факт на тестовой выборке', fontsize=14)
plt.tight_layout()
plt.show()

### Выводы по моделям

- **LightGBM — лучшая модель** по всем метрикам (MAE=0.057, WAPE=26.5%).
- Ridge заметно хуже: линейной зависимости недостаточно.
- Directional Accuracy ~0.45 у всех моделей на валидации — направление изменений улавливается слабо; все модели «срезают» экстремумы.

---
## 2.3 Оценка качества моделей

In [ ]:
# ------------------------------------------------------------
# 5.1 Анализ остатков лучшей модели (LightGBM)
# ------------------------------------------------------------

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf

residuals = y_test.values - y_pred_lgb

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0,0].plot(y_test.index, residuals, 'o-', markersize=2, linewidth=0.7)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Остатки во времени')
axes[0,0].set_ylabel('Residual')
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

axes[0,1].hist(residuals, bins=40, density=True, alpha=0.7, edgecolor='black')
xr = np.linspace(residuals.min(), residuals.max(), 200)
axes[0,1].plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()),
               'r-', linewidth=2, label='N(μ,σ)')
axes[0,1].set_title('Распределение остатков')
axes[0,1].legend()

sm.qqplot(residuals, line='s', ax=axes[0,2], alpha=0.5)
axes[0,2].set_title('Q-Q plot (нормальность)')

plot_acf(residuals, lags=30, ax=axes[1,0], alpha=0.05)
axes[1,0].set_title('ACF остатков (автокорреляция)')

axes[1,1].scatter(y_pred_lgb, residuals, s=10, alpha=0.5)
axes[1,1].axhline(0, color='red', linestyle='--')
axes[1,1].set_xlabel('Предсказанное значение')
axes[1,1].set_ylabel('Остаток')
axes[1,1].set_title('Остатки vs Прогноз (гетероскедастичность)')

axes[1,2].scatter(y_test.values, residuals, s=10, alpha=0.5)
axes[1,2].axhline(0, color='red', linestyle='--')
axes[1,2].set_xlabel('Факт')
axes[1,2].set_ylabel('Остаток')
axes[1,2].set_title('Остатки vs Факт')

plt.suptitle('Анализ остатков — LightGBM', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 5.2 Статистические тесты на остатки
# ------------------------------------------------------------

from statsmodels.stats.diagnostic import het_breuschpagan

_, p_shapiro = stats.shapiro(residuals[:500])
jb_stat, p_jb, _, _ = sm.stats.stattools.jarque_bera(residuals)
dw = durbin_watson(residuals)

X_test_sm = sm.add_constant(y_pred_lgb)
_, p_bp, _, _ = het_breuschpagan(residuals, X_test_sm)

print('=== Статистические тесты на остатки (LightGBM) ===')
print(f'Shapiro-Wilk  p-value: {p_shapiro:.4f}  {"✓ норм." if p_shapiro > 0.05 else "✗ не норм."}')
print(f'Jarque-Bera   p-value: {p_jb:.4f}  {"✓ норм." if p_jb > 0.05 else "✗ не норм."}')
print(f'Durbin-Watson:         {dw:.4f}  {"✓ нет автокорр." if 1.5 < dw < 2.5 else "✗ есть автокорр."}')
print(f'Breusch-Pagan p-value: {p_bp:.4f}  {"✓ гомоскед." if p_bp > 0.05 else "✗ гетероскед."}')

In [ ]:
# ------------------------------------------------------------
# 5.3 SHAP — важность и направление влияния признаков
# ------------------------------------------------------------

explainer   = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, max_display=20, show=False, plot_type='bar')
plt.title('SHAP — важность признаков (LightGBM)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, max_display=20, show=False)
plt.title('SHAP beeswarm — направление влияния признаков')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 5.4 Permutation Importance
# ------------------------------------------------------------

from sklearn.inspection import permutation_importance

perm = permutation_importance(
    lgb_model, X_test, y_test,
    n_repeats=10, random_state=42,
    scoring='neg_mean_absolute_error'
)

perm_df = pd.DataFrame({
    'feature':          feature_cols,
    'importance_mean':  perm.importances_mean,
    'importance_std':   perm.importances_std
}).sort_values('importance_mean', ascending=False).head(20)

print('=== Топ-20 по Permutation Importance ===')
print(perm_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
perm_df.sort_values('importance_mean').plot(
    kind='barh', x='feature', y='importance_mean',
    xerr='importance_std', ax=ax, legend=False
)
ax.set_title('Permutation Importance — LightGBM (топ-20)')
ax.set_xlabel('Снижение MAE при перемешивании')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 5.5 Финальная проверка на внешнем тесте (data_test_small)
# ------------------------------------------------------------

TELE_COLS_TEST = [f'telemetry_{i}' for i in range(12)]

data_test_clean = data_test[TELE_COLS_TEST].interpolate(method='linear', limit=10)
for col in TELE_COLS_TEST:
    q_lo = data_train_clean[col].quantile(0.005)
    q_hi = data_train_clean[col].quantile(0.995)
    data_test_clean[col] = data_test_clean[col].clip(q_lo, q_hi)
data_test_clean['telemetry_4'] = data_test_clean['telemetry_4'].fillna(median_t4)

rolling_frames_test = []
for col in KEY_COLS:
    series = data_test_clean[col]
    for win_name, win_size in WINDOWS.items():
        roll = series.rolling(window=win_size, min_periods=win_size//2)
        rolling_frames_test.append(roll.mean().rename(f'{col}_rmean_{win_name}'))
        rolling_frames_test.append(roll.std().rename(f'{col}_rstd_{win_name}'))
        rolling_frames_test.append(roll.min().rename(f'{col}_rmin_{win_name}'))
        rolling_frames_test.append(roll.max().rename(f'{col}_rmax_{win_name}'))
    rolling_frames_test.append(series.diff(1).rename(f'{col}_diff1'))
    rolling_frames_test.append(series.diff(10).rename(f'{col}_diff10'))
    rolling_frames_test.append(series.pct_change(30).replace(
        [np.inf, -np.inf], np.nan).rename(f'{col}_pct30'))

df_rolling_test = pd.concat(rolling_frames_test, axis=1)

target_test_shifted = target_test.copy()
target_test_shifted.index = target_test.index - pd.Timedelta(minutes=LAG_MINUTES)

tele_test_reset   = pd.concat([data_test_clean, df_rolling_test], axis=1).reset_index()
target_test_reset = target_test_shifted.reset_index()

df_test_full = pd.merge_asof(
    target_test_reset.sort_values('datetime'),
    tele_test_reset.sort_values('datetime'),
    on='datetime', direction='backward',
    tolerance=pd.Timedelta('5min')
).set_index('datetime')

for col in KEY_COLS:
    for lag in [1, 2, 3, 6]:
        df_test_full[f'{col}_lag{lag}'] = df_test_full[col].shift(lag)

last_targets = list(y_test.values[-3:])
df_test_full['target_lag1'] = np.nan
df_test_full['target_lag2'] = np.nan
df_test_full['target_lag3'] = np.nan
if len(df_test_full) >= 1:
    df_test_full.iloc[0, df_test_full.columns.get_loc('target_lag1')] = last_targets[-1]
    df_test_full.iloc[0, df_test_full.columns.get_loc('target_lag2')] = last_targets[-2]
    df_test_full.iloc[0, df_test_full.columns.get_loc('target_lag3')] = last_targets[-3]

df_test_full['hour']        = df_test_full.index.hour
df_test_full['day_of_week'] = df_test_full.index.dayofweek
df_test_full['month']       = df_test_full.index.month
df_test_full['hour_sin']    = np.sin(2 * np.pi * df_test_full['hour'] / 24)
df_test_full['hour_cos']    = np.cos(2 * np.pi * df_test_full['hour'] / 24)

# Выравниваем колонки по feature_cols из train
for col in set(feature_cols) - set(df_test_full.columns):
    df_test_full[col] = np.nan

X_ext = df_test_full[feature_cols].dropna(how='all')
y_ext = df_test_full.loc[X_ext.index, 'target']

for col in feature_cols:
    if X_ext[col].isnull().any():
        X_ext[col] = X_ext[col].fillna(X_train[col].median())

print(f'X_ext shape: {X_ext.shape}, NaN: {X_ext.isnull().sum().sum()}')

y_pred_ext  = lgb_model.predict(X_ext)
results_ext = evaluate('LightGBM (external test_small)', y_ext, y_pred_ext)

In [ ]:
# ------------------------------------------------------------
# 5.6 Итоговое сравнение всех моделей
# ------------------------------------------------------------

results_all = pd.DataFrame([results_ridge, results_lgb,
                             results_xgb,   results_rf]).set_index('model').round(4)
print('\n=== ФИНАЛЬНАЯ СВОДНАЯ ТАБЛИЦА (валидационная выборка) ===')
print(results_all.to_string())
print('\n=== ВНЕШНИЙ ТЕСТ (test_small) — LightGBM ===')
print(f'  MAE:  {results_ext["MAE"]:.4f}')
print(f'  RMSE: {results_ext["RMSE"]:.4f}')
print(f'  WAPE: {results_ext["WAPE"]:.4f}')
print(f'  DA:   {results_ext["DA"]:.4f}')

---
## 2.4 Документирование и интерпретация — итоговые выводы

### Результаты

| Метрика | Валидация (LightGBM) | Внешний тест |
|---------|---------------------|--------------|
| MAE  | 0.0567 | 0.0656 |
| RMSE | 0.0689 | 0.0794 |
| WAPE | 26.5%  | 31.3%  |
| DA   | 45.3%  | **55.5%** |

### Интерпретация физических закономерностей

1. **Инерционность процесса:** `target_lag1` — доминирующий предиктор (SHAP в 14× сильнее второго). Концентрация продукта определяется предыдущим состоянием печи — физически обоснованно для термохимических реакций.
2. **telemetry_0** — ключевой датчик интенсивности процесса; его rolling-статистики за 30 мин важнее мгновенного значения.
3. **Скользящие минимумы telemetry_9 и telemetry_10** на длинных окнах (2–12ч) — устойчиво низкий уровень этих показателей предсказывает изменение концентрации.
4. **Сезонный признак `month`** входит в топ-20 SHAP — режим работы печи различается по месяцам.

### Ограничения и направления улучшения

- Модель недооценивает пиковые значения (>0.4) — именно нештатные ситуации. Рекомендуется отдельный бинарный классификатор аномалий.
- Автокорреляция остатков (DW=1.41) — неполнота признакового пространства; не все физические факторы процесса отражены в 16 датчиках.
- Гетероскедастичность: точность падает при высоких концентрациях — модель сильнее ошибается именно тогда, когда ошибка наиболее критична.
- telemetry_12–15 (появляются в конце периода) не использовались — по мере накопления данных могут существенно улучшить качество.